In [1]:
import pandas as pd
import pulp

In [2]:
df_nutrition = pd.read_csv('dataset_gizi.csv')
df_nutrition.set_index('KOMODITAS', inplace=True)

predicted_prices_tomorrow = {
    'Beras Medium': 13500,
    'Gula Pasir Curah': 17500,
    'Minyak Goreng Sawit Curah': 15500,
    'Daging Sapi Paha Belakang': 125000,
    'Daging Ayam Ras': 38200,
    'Telur Ayam Ras': 29500,
    'Tepung Terigu': 11000,
    'Kedelai Impor': 12000,
    'Cabai Merah Keriting': 45000,
    'Bawang Merah': 35000
}

available_items = [item for item in predicted_prices_tomorrow.keys() if item in df_nutrition.index]

In [3]:
# Membuat model untuk meminimalkan biaya (LpMinimize)
lp_problem = pulp.LpProblem("NutriCost_Diet_Optimization", pulp.LpMinimize)

# Mendefinisikan Variabel Keputusan "Berapa Kg bahan X yang harus dibeli?"
# lowBound=0 memastikan tidak membeli bahan dalam jumlah minus
x = pulp.LpVariable.dicts("qty_kg", available_items, lowBound=0, cat='Continuous')

In [4]:
# 1. FUNGSI OBJEKTIF: Total Biaya = Harga * Kuantitas
lp_problem += pulp.lpSum([predicted_prices_tomorrow[i] * x[i] for i in available_items]), "Total_Cost"

# 2. KENDALA GIZI 1: Total Energi Minimal 2000 Kkal
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Energi'] * x[i] for i in available_items]) >= 2000, "Min_Energy"

# 3. KENDALA GIZI 2: Total Protein Minimal 50 gram (Cegah Stunting)
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Protein'] * x[i] for i in available_items]) >= 50, "Min_Protein"

# 4. KENDALA LOGIKA MANUSIA (Batas maksimal makan per hari per anak)
for i in available_items:
    if 'Beras' in i:
        lp_problem += x[i] <= 0.300  # Maksimal 300g beras sehari
    elif 'Gula' in i:
        lp_problem += x[i] <= 0.020  # Maksimal 20g gula sehari (batas sehat)
    elif 'Minyak' in i:
        lp_problem += x[i] <= 0.030  # Maksimal 30g minyak goreng
    elif 'Ayam' in i or 'Sapi' in i or 'Telur' in i:
        lp_problem += x[i] <= 0.150  # Maksimal 150g sumber protein hewani
    elif 'Cabai' in i or 'Bawang' in i:
        lp_problem += x[i] <= 0.010  # Maksimal 10g 

In [5]:
lp_problem.solve()

# Cek apakah mesin berhasil menemukan jalan keluar
if pulp.LpStatus[lp_problem.status] == 'Optimal':
    total_cost = pulp.value(lp_problem.objective)
    
    print(f"Total Biaya per Anak : Rp {total_cost:,.2f}")
    print("\nRekomendasi Resep & Gramatur")
    
    total_kcal = 0
    total_protein = 0
    
    for item in available_items:
        qty_kg = x[item].varValue
        if qty_kg > 0.001:  # Hanya tampilkan bahan yang dipakai lebih dari 1 gram
            qty_grams = qty_kg * 1000
            kcal_supplied = qty_kg * df_nutrition.loc[item, 'Energi']
            protein_supplied = qty_kg * df_nutrition.loc[item, 'Protein']
            cost_item = qty_kg * predicted_prices_tomorrow[item]
            
            print(f"- {item:.<30} {qty_grams:>5.0f} gram  (Rp {cost_item:>6,.0f})")
            
            total_kcal += kcal_supplied
            total_protein += protein_supplied
            
    print("\n[ Pencapaian Target Gizi ]")
    print(f"- Total Energi Terpenuhi   : {total_kcal:.1f} Kcal")
    print(f"- Total Protein Terpenuhi  : {total_protein:.1f} gram")
else:
    print("ERROR: Tidak dapat menemukan menu optimal")

Total Biaya per Anak : Rp 6,230.94

Rekomendasi Resep & Gramatur
- Minyak Goreng Sawit Curah.....    30 gram  (Rp    465)
- Tepung Terigu.................   509 gram  (Rp  5,600)
- Kedelai Impor.................    14 gram  (Rp    166)

[ Pencapaian Target Gizi ]
- Total Energi Terpenuhi   : 2000.0 Kcal
- Total Protein Terpenuhi  : 50.0 gram
